# Alpha Research Notebook — Hyperliquid Seconds / Mid-Frequency

Objectif : détecter des signaux candidats alpha sur crypto perps à partir de données secondes :
order book L2, trades récents, VWAP, microprice, order-flow imbalance, décorrélation BTC, régime de volatilité et coûts.

Un signal n'est pas un alpha tant qu'il ne vérifie pas :
1. pouvoir prédictif sur le retour futur,
2. robustesse sur plusieurs actifs,
3. stabilité temporelle,
4. survie après frais/slippage,
5. absence de simple réplication BTC/ETH beta,
6. absence d'overfit évident.

In [ ]:
import os
import math
import warnings
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

ROOT = Path.cwd()
LOG_DIR = ROOT / "logs"
REPORT_DIR = ROOT / "reports"
REPORT_DIR.mkdir(exist_ok=True)

print("ROOT:", ROOT)

## 1. Load data

Fichier idéal à produire depuis ton moteur :

`logs/seconds_features.csv`

Colonnes recommandées :
`ts, symbol, mid, best_bid, best_ask, spread_bps, obi_1, obi_3, obi_5, obi_10,
trade_imbalance_5s, trade_imbalance_10s, trade_imbalance_30s,
buy_volume_usd_10s, sell_volume_usd_10s, vwap_5s, vwap_15s, vwap_30s,
microprice, microprice_pressure, r_5s, r_15s, r_30s, rv_30s, rv_60s`

In [ ]:
def load_csv_if_exists(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f"Missing: {path}")
        return pd.DataFrame()
    df = pd.read_csv(path)
    print(f"Loaded {path}: {df.shape}")
    return df

features = load_csv_if_exists(LOG_DIR / "seconds_features.csv")

if features.empty:
    features = pd.DataFrame(columns=[
        "ts", "symbol", "mid", "best_bid", "best_ask", "spread_bps",
        "obi_1", "obi_3", "obi_5", "obi_10",
        "trade_imbalance_5s", "trade_imbalance_10s", "trade_imbalance_30s",
        "buy_volume_usd_10s", "sell_volume_usd_10s",
        "vwap_5s", "vwap_15s", "vwap_30s",
        "microprice", "microprice_pressure",
        "r_5s", "r_15s", "r_30s", "rv_30s", "rv_60s"
    ])

features.head()

In [ ]:
def normalize_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if df.empty:
        return df
    if "ts" in df.columns:
        if pd.api.types.is_numeric_dtype(df["ts"]):
            df["datetime"] = pd.to_datetime(df["ts"], unit="s", utc=True)
        else:
            df["datetime"] = pd.to_datetime(df["ts"], utc=True, errors="coerce")
    elif "datetime" in df.columns:
        df["datetime"] = pd.to_datetime(df["datetime"], utc=True, errors="coerce")
        df["ts"] = df["datetime"].astype("int64") / 1e9
    else:
        raise ValueError("features must contain ts or datetime")
    df["symbol"] = df["symbol"].astype(str).str.upper()
    df = df.dropna(subset=["datetime", "symbol"])
    return df.sort_values(["symbol", "datetime"]).reset_index(drop=True)

features = normalize_features(features)
features.head()

## 2. Forward returns

On calcule les retours futurs sur plusieurs horizons : 5s, 15s, 30s, 60s, 120s, 300s.

In [ ]:
HORIZONS = [5, 15, 30, 60, 120, 300]

def add_forward_returns(df: pd.DataFrame, horizons=HORIZONS) -> pd.DataFrame:
    df = df.copy()
    if df.empty or "mid" not in df.columns:
        return df
    out = []
    for sym, g in df.groupby("symbol", sort=False):
        g = g.sort_values("datetime").copy()
        for h in horizons:
            g[f"fwd_mid_{h}s"] = g["mid"].shift(-h)
            g[f"fwd_ret_{h}s"] = g[f"fwd_mid_{h}s"] / g["mid"] - 1.0
            g[f"fwd_ret_{h}s_bps"] = g[f"fwd_ret_{h}s"] * 10_000
        out.append(g)
    return pd.concat(out, ignore_index=True) if out else df

features = add_forward_returns(features)
features.filter(regex="fwd_ret").head()

## 3. Candidate alpha features

Signaux classiques :
- order book imbalance,
- trade imbalance,
- VWAP slope,
- microprice pressure,
- short momentum.

Signaux plus différenciants :
- divergence book-flow,
- absorption proxy,
- pressure persistence,
- imbalance acceleration,
- liquidity vacuum,
- BTC residualization.

In [ ]:
def safe_zscore(s: pd.Series, window: int = 300) -> pd.Series:
    m = s.rolling(window, min_periods=max(10, window // 10)).mean()
    sd = s.rolling(window, min_periods=max(10, window // 10)).std()
    return (s - m) / sd.replace(0, np.nan)

def add_candidate_alpha_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if df.empty:
        return df

    required = ["obi_1", "obi_3", "obi_5", "obi_10", "trade_imbalance_10s",
                "vwap_5s", "vwap_30s", "microprice_pressure", "r_5s", "r_15s",
                "spread_bps", "rv_30s"]
    for col in required:
        if col not in df.columns:
            df[col] = np.nan

    out = []
    for sym, g in df.groupby("symbol", sort=False):
        g = g.sort_values("datetime").copy()
        g["vwap_slope_5_30"] = g["vwap_5s"] / g["vwap_30s"] - 1.0
        g["obi5_delta_1s"] = g["obi_5"].diff(1)
        g["obi5_delta_5s"] = g["obi_5"].diff(5)
        g["obi5_pos_persistence_10s"] = (g["obi_5"] > 0).rolling(10, min_periods=3).mean()
        g["obi5_neg_persistence_10s"] = (g["obi_5"] < 0).rolling(10, min_periods=3).mean()
        g["book_flow_alignment"] = np.sign(g["obi_5"]) * np.sign(g["trade_imbalance_10s"])
        g["book_flow_divergence"] = g["obi_5"] - g["trade_imbalance_10s"]
        g["absorption_sell_proxy"] = np.maximum(g["trade_imbalance_10s"], 0) * np.maximum(-g["r_5s"], 0)
        g["absorption_buy_proxy"] = np.maximum(-g["trade_imbalance_10s"], 0) * np.maximum(g["r_5s"], 0)
        g["liquidity_vacuum"] = safe_zscore(g["spread_bps"], 300) + safe_zscore(g["rv_30s"], 300)
        g["pressure_score_raw"] = (
            0.25 * g["obi_5"].fillna(0)
            + 0.25 * g["trade_imbalance_10s"].fillna(0)
            + 0.20 * np.tanh(g["vwap_slope_5_30"].fillna(0) * 10_000 / 10)
            + 0.15 * np.tanh(g["microprice_pressure"].fillna(0) * 10_000 / 5)
            + 0.15 * np.tanh(g["r_5s"].fillna(0) * 10_000 / 5)
        )
        g["pressure_score_z"] = safe_zscore(g["pressure_score_raw"], 300)
        out.append(g)
    return pd.concat(out, ignore_index=True)

features = add_candidate_alpha_features(features)
features.filter(regex="pressure|divergence|absorption|vacuum|alignment").head()

## 4. BTC residualization

Un faux alpha crypto prédit souvent juste le bêta BTC.  
Ici on enlève l'effet BTC pour tester si le signal prédit un retour résiduel.

In [ ]:
def add_btc_residual_returns(df: pd.DataFrame, horizon: int = 60, beta_window: int = 600) -> pd.DataFrame:
    df = df.copy()
    if df.empty or "BTC" not in set(df["symbol"]):
        print("BTC not available; skipping residualization.")
        return df
    ret_col = f"fwd_ret_{horizon}s"
    if ret_col not in df.columns:
        return df

    btc = df[df["symbol"] == "BTC"][["datetime", ret_col]].rename(columns={ret_col: f"btc_{ret_col}"})
    df = df.merge(btc, on="datetime", how="left")
    out = []
    for sym, g in df.groupby("symbol", sort=False):
        g = g.sort_values("datetime").copy()
        if sym == "BTC":
            g[f"resid_ret_{horizon}s"] = 0.0
            out.append(g)
            continue
        x = g[f"btc_{ret_col}"]
        y = g[ret_col]
        cov = y.rolling(beta_window, min_periods=100).cov(x)
        var = x.rolling(beta_window, min_periods=100).var()
        beta = cov / var.replace(0, np.nan)
        g[f"beta_btc_{horizon}s"] = beta
        g[f"resid_ret_{horizon}s"] = y - beta * x
        g[f"resid_ret_{horizon}s_bps"] = g[f"resid_ret_{horizon}s"] * 10_000
        out.append(g)
    return pd.concat(out, ignore_index=True)

features = add_btc_residual_returns(features, horizon=60)
features.filter(regex="beta|resid").head()

## 5. Alpha diagnostics

Tests :
- Pearson IC,
- Spearman IC,
- mean forward return,
- decay par horizon,
- robustesse par symbole.

In [ ]:
CANDIDATE_SIGNALS = [
    "obi_5", "trade_imbalance_10s", "vwap_slope_5_30",
    "microprice_pressure", "obi5_delta_1s", "obi5_delta_5s",
    "book_flow_alignment", "book_flow_divergence",
    "absorption_sell_proxy", "absorption_buy_proxy",
    "liquidity_vacuum", "pressure_score_raw", "pressure_score_z"
]

def compute_ic_table(df: pd.DataFrame, signals=CANDIDATE_SIGNALS, horizons=HORIZONS) -> pd.DataFrame:
    rows = []
    if df.empty:
        return pd.DataFrame()
    for sig in signals:
        if sig not in df.columns:
            continue
        for h in horizons:
            ret_col = f"fwd_ret_{h}s_bps"
            if ret_col not in df.columns:
                continue
            tmp = df[[sig, ret_col, "symbol"]].replace([np.inf, -np.inf], np.nan).dropna()
            if len(tmp) < 100:
                continue
            rows.append({
                "signal": sig,
                "horizon_s": h,
                "n": len(tmp),
                "pearson_ic": tmp[sig].corr(tmp[ret_col], method="pearson"),
                "spearman_ic": tmp[sig].corr(tmp[ret_col], method="spearman"),
                "mean_fwd_bps": tmp[ret_col].mean(),
                "std_fwd_bps": tmp[ret_col].std(),
            })
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values(["horizon_s", "spearman_ic"], ascending=[True, False])

ic_table = compute_ic_table(features)
ic_table.head(30)

## 6. Bucket analysis

Un bon alpha doit montrer une relation ordonnée entre bucket de signal et retour futur.

In [ ]:
def bucket_analysis(df: pd.DataFrame, signal: str, ret_col: str, q: int = 10) -> pd.DataFrame:
    tmp = df[[signal, ret_col, "symbol"]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(tmp) < q * 20:
        return pd.DataFrame()
    tmp["bucket"] = pd.qcut(tmp[signal], q=q, labels=False, duplicates="drop")
    return tmp.groupby("bucket").agg(
        n=(ret_col, "size"),
        signal_mean=(signal, "mean"),
        fwd_ret_mean_bps=(ret_col, "mean"),
        fwd_ret_median_bps=(ret_col, "median"),
        hit_rate=(ret_col, lambda x: (x > 0).mean()),
    ).reset_index()

bucket_res = bucket_analysis(features, "pressure_score_raw", "fwd_ret_60s_bps", q=10)
bucket_res

## 7. Threshold alpha simulation

Test rapide :
- long si signal > quantile haut,
- short si signal < quantile bas,
- filtre spread,
- coûts round-trip.

In [ ]:
@dataclass
class AlphaSimConfig:
    signal: str
    horizon_s: int = 60
    upper_q: float = 0.90
    lower_q: float = 0.10
    max_spread_bps: float = 8.0
    round_trip_cost_bps: float = 12.0

def simulate_threshold_alpha(df: pd.DataFrame, cfg: AlphaSimConfig) -> pd.DataFrame:
    ret_col = f"fwd_ret_{cfg.horizon_s}s_bps"
    needed = [cfg.signal, ret_col, "symbol", "datetime"]
    if "spread_bps" in df.columns:
        needed.append("spread_bps")
    tmp = df[needed].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if tmp.empty:
        return pd.DataFrame()

    trades = []
    for sym, g in tmp.groupby("symbol", sort=False):
        g = g.sort_values("datetime").copy()
        upper = g[cfg.signal].quantile(cfg.upper_q)
        lower = g[cfg.signal].quantile(cfg.lower_q)
        if "spread_bps" in g.columns:
            g = g[g["spread_bps"] <= cfg.max_spread_bps]

        longs = g[g[cfg.signal] >= upper].copy()
        longs["side"] = "LONG"
        longs["gross_bps"] = longs[ret_col]

        shorts = g[g[cfg.signal] <= lower].copy()
        shorts["side"] = "SHORT"
        shorts["gross_bps"] = -shorts[ret_col]

        x = pd.concat([longs, shorts], ignore_index=True)
        x["net_bps"] = x["gross_bps"] - cfg.round_trip_cost_bps
        trades.append(x)
    return pd.concat(trades, ignore_index=True) if trades else pd.DataFrame()

def summarize_trades(trades: pd.DataFrame) -> pd.DataFrame:
    if trades.empty:
        return pd.DataFrame([{"n": 0}])
    losses = trades.loc[trades["net_bps"] < 0, "net_bps"]
    wins = trades.loc[trades["net_bps"] > 0, "net_bps"]
    pf = wins.sum() / abs(losses.sum()) if len(losses) else np.inf
    return pd.DataFrame([{
        "n": len(trades),
        "gross_mean_bps": trades["gross_bps"].mean(),
        "net_mean_bps": trades["net_bps"].mean(),
        "net_median_bps": trades["net_bps"].median(),
        "win_rate_net": (trades["net_bps"] > 0).mean(),
        "profit_factor_net": pf,
        "avg_win_bps": wins.mean(),
        "avg_loss_bps": losses.mean(),
    }])

sim_trades = simulate_threshold_alpha(features, AlphaSimConfig(signal="pressure_score_raw", horizon_s=60))
summarize_trades(sim_trades)

## 8. Scan all candidate signals

In [ ]:
def scan_signals(df: pd.DataFrame, signals=CANDIDATE_SIGNALS, horizons=[15, 30, 60, 120], costs=[8, 12, 16]):
    rows = []
    for sig in signals:
        if sig not in df.columns:
            continue
        for h in horizons:
            for cost in costs:
                tr = simulate_threshold_alpha(df, AlphaSimConfig(signal=sig, horizon_s=h, round_trip_cost_bps=cost))
                if tr.empty or len(tr) < 50:
                    continue
                s = summarize_trades(tr).iloc[0].to_dict()
                s.update({"signal": sig, "horizon_s": h, "cost_bps": cost})
                rows.append(s)
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values(["net_mean_bps", "profit_factor_net"], ascending=False)

scan = scan_signals(features)
scan.head(30)

## 9. Walk-forward validation

Calibration sur train, test hors-échantillon.

In [ ]:
def walk_forward_threshold_alpha(
    df: pd.DataFrame,
    signal: str,
    horizon_s: int = 60,
    train_days: int = 3,
    test_days: int = 1,
    upper_q: float = 0.90,
    lower_q: float = 0.10,
    cost_bps: float = 12.0,
    max_spread_bps: float = 8.0,
) -> pd.DataFrame:
    ret_col = f"fwd_ret_{horizon_s}s_bps"
    needed = ["datetime", "symbol", signal, ret_col, "spread_bps"]
    tmp = df[needed].dropna().copy()
    if tmp.empty:
        return pd.DataFrame()
    tmp = tmp.sort_values("datetime")
    start = tmp["datetime"].min()
    end = tmp["datetime"].max()

    rows = []
    cur = start
    train_delta = pd.Timedelta(days=train_days)
    test_delta = pd.Timedelta(days=test_days)

    while cur + train_delta + test_delta <= end:
        train_start = cur
        train_end = cur + train_delta
        test_end = train_end + test_delta
        train = tmp[(tmp["datetime"] >= train_start) & (tmp["datetime"] < train_end)]
        test = tmp[(tmp["datetime"] >= train_end) & (tmp["datetime"] < test_end)]

        for sym, train_g in train.groupby("symbol"):
            test_g = test[test["symbol"] == sym].copy()
            if len(train_g) < 50 or len(test_g) < 20:
                continue
            upper = train_g[signal].quantile(upper_q)
            lower = train_g[signal].quantile(lower_q)
            test_g = test_g[test_g["spread_bps"] <= max_spread_bps]

            long = test_g[test_g[signal] >= upper].copy()
            long["gross_bps"] = long[ret_col]
            short = test_g[test_g[signal] <= lower].copy()
            short["gross_bps"] = -short[ret_col]
            trades = pd.concat([long, short], ignore_index=True)
            if trades.empty:
                continue
            trades["net_bps"] = trades["gross_bps"] - cost_bps
            losses = trades.loc[trades["net_bps"] < 0, "net_bps"]
            wins = trades.loc[trades["net_bps"] > 0, "net_bps"]
            rows.append({
                "train_start": train_start,
                "train_end": train_end,
                "test_end": test_end,
                "symbol": sym,
                "n": len(trades),
                "net_mean_bps": trades["net_bps"].mean(),
                "win_rate": (trades["net_bps"] > 0).mean(),
                "profit_factor": wins.sum() / abs(losses.sum()) if len(losses) else np.inf,
            })
        cur += test_delta
    return pd.DataFrame(rows)

wf = walk_forward_threshold_alpha(features, signal="pressure_score_raw", horizon_s=60)
wf.head()

## 10. Uniqueness test

On vérifie si le signal reste utile après contrôle de variables banales :
momentum court terme, spread, volatilité, OBI, trade imbalance.

In [ ]:
def residualize(y: pd.Series, X: pd.DataFrame) -> pd.Series:
    tmp = pd.concat([y, X], axis=1).replace([np.inf, -np.inf], np.nan).dropna()
    if tmp.empty or len(tmp) < X.shape[1] + 10:
        return pd.Series(index=y.index, dtype=float)
    yv = tmp.iloc[:, 0].values
    Xv = tmp.iloc[:, 1:].values
    Xv = np.column_stack([np.ones(len(Xv)), Xv])
    beta = np.linalg.lstsq(Xv, yv, rcond=None)[0]
    out = pd.Series(index=tmp.index, data=yv - Xv @ beta)
    return out.reindex(y.index)

def uniqueness_test(df: pd.DataFrame, signal: str, horizon_s: int = 60) -> dict:
    ret_col = f"fwd_ret_{horizon_s}s_bps"
    controls = [c for c in ["r_5s", "r_15s", "spread_bps", "rv_30s", "trade_imbalance_10s", "obi_5"] if c in df.columns and c != signal]
    tmp = df[[signal, ret_col] + controls].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(tmp) < 200:
        return {"signal": signal, "horizon_s": horizon_s, "n": len(tmp)}
    raw_ic = tmp[signal].corr(tmp[ret_col], method="spearman")
    resid_signal = residualize(tmp[signal], tmp[controls])
    resid_ic = resid_signal.corr(tmp[ret_col], method="spearman")
    return {
        "signal": signal,
        "horizon_s": horizon_s,
        "n": len(tmp),
        "raw_spearman_ic": raw_ic,
        "residual_spearman_ic": resid_ic,
        "controls": ",".join(controls),
    }

unique_rows = [uniqueness_test(features, s, 60) for s in CANDIDATE_SIGNALS if s in features.columns]
pd.DataFrame(unique_rows).sort_values("residual_spearman_ic", ascending=False).head(20)

## 11. Generate Markdown report

In [ ]:
def generate_alpha_report(scan: pd.DataFrame, ic_table: pd.DataFrame, output_path: Path):
    lines = []
    lines.append("# Alpha Research Report\n")
    lines.append(f"Generated: {pd.Timestamp.utcnow()}\n")
    lines.append("## Top threshold simulations\n")
    lines.append(scan.head(20).to_markdown(index=False) if not scan.empty else "No scan results available.")
    lines.append("\n\n## Top IC results\n")
    lines.append(ic_table.sort_values("spearman_ic", ascending=False).head(30).to_markdown(index=False) if not ic_table.empty else "No IC results available.")
    lines.append("\n\n## Validation checklist\n")
    lines.append("""
A signal is not an alpha unless:
- it survives transaction costs,
- it works out-of-sample,
- it is robust across symbols,
- it is not just BTC beta,
- it is not just spread/liquidity/volatility effect,
- it has a plausible microstructure explanation,
- it does not rely on impossible fills.
""")
    output_path.write_text("\n".join(lines), encoding="utf-8")
    return output_path

report_path = REPORT_DIR / "alpha_research_report.md"
generate_alpha_report(scan if "scan" in globals() else pd.DataFrame(), ic_table if "ic_table" in globals() else pd.DataFrame(), report_path)
print("Report written:", report_path)